In [ ]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import random
import math

import pandas as pd


import copy

import cvxpy
cp = cvxpy

import figurefirst as fifi

from braid_analysis import braid_analysis_plots

In [ ]:
from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis import unifying_algo_plots as uap

# Helper Functions

In [ ]:
#from unifying_algo_analysis_helper import *
from splitflow.set_zorder_functions import *

In [ ]:
def mean_angle(angle):
    
    mean = np.arctan2( np.nanmean(np.sin(angle)), np.nanmean(np.cos(angle)) )
    return mean

def angle_distance(angle1, angle2):
    """
    Calculate the minimum distance between two angles.

    Parameters:
    -----------
    angle1, angle2 : float or array-like
        Angles in radians

    Returns:
    --------
    float or array
        Minimum distance between angles in radians 
        Range: [-π, π]
    """
    diff = angle1 - angle2
    # Wrap to [-π, π]
    distance = np.arctan2(np.sin(diff), np.cos(diff))
    return distance

def parametric_model(unifying_algo_data, ix_master):
    """
    Model course vs. aligned time built from the median fit parameters (the pink line), instead of the circular
    mean of all aligned affine fits. Uses each trajectory's best fit (lowest rmse_affine), normalized to a positive
    slope as in uaa.get_aligned_course_array (mirror: flip the sign of the off-diagonal AM terms and of tx).
    Model: phi = median|slope| * aligned_ix (phase 0 at aligned t=0, a 0-crossing of the linear fit),
           course = atan2(w0, w1), w = median(AM) @ [sin(phi), cos(phi)] + median([tx, ty])   ((sin, cos) convention)
    """
    best = (unifying_algo_data.sort_values('rmse_affine', kind='stable')
            .groupby('obj_id_unique_event', sort=False).head(1))
    flip = np.diag([-1.0, 1.0])
    slopes, AMs, ts = [], [], []
    for _, row in best.iterrows():
        AM = np.array([[row.AM_0_0, row.AM_0_1], [row.AM_1_0, row.AM_1_1]])
        t = np.nan_to_num(np.array([row.tx, row.ty], dtype=float))  # fits without translation store None
        if row.slope < 0:
            AM, t = flip @ AM @ flip, flip @ t
        slopes.append(abs(row.slope)); AMs.append(AM); ts.append(t)
    phi = np.median(slopes) * np.asarray(ix_master, dtype=float)
    w = np.median(AMs, axis=0) @ np.vstack((np.sin(phi), np.cos(phi))) + np.median(ts, axis=0)[:, None]
    return np.arctan2(w[0], w[1])


In [ ]:
def clean_labels(ax, show_labels, spines=['left', 'bottom']):
    #set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
    #set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)
    ax.set_rasterization_zorder(0)

    ax.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_ylim(-np.pi, np.pi)
    ax.set_xlim(-2, 2)
    ax.set_xticks([-200, -100, 0, 100, 200])

    if show_labels:
        ax.set_xticklabels(['-2', '-1', '0', '1', '2'])
    else:
        ax.set_xticklabels([])
    
    if show_labels:
        ax.set_yticklabels(['$-\pi$', '','$0$','','$\pi$',])
    else:
        ax.set_yticklabels([])
        
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'])
    
    if show_labels:
        ax.set_ylabel('Course direction', labelpad=-2)
        ax.set_xlabel('Aligned time (s)', labelpad=1)
    else:
        ax.set_ylabel('')
        ax.set_xlabel('')
    
    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)
    
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)

In [ ]:
FIGURE_NAME = 'unifying_analysis.svg'

In [ ]:
TRANSLATION = True

In [ ]:
COURSE_MARKER_SIZE = 2
COURSE_ALPHA_MULTIPLIER = 3

# The pink model line is drawn from this aligned time on (ix units: 100 = 1 s)
MODEL_START_IX = -100  # -1 s


In [ ]:
if TRANSLATION:
    FIGURE_NAME = 'unifying_analysis_translationTrue.svg'

In [ ]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

# With real laminar data

In [ ]:
show_labels = False

In [ ]:
df = pd.read_hdf('../../../../Data/Experimental_Fly_Data/flies_laminar_c1xwt_preprocessed_optotrigger_trimmed.hdf')
fname = 'laminar_data_flash_translation' + str(TRANSLATION) + '.parquet'
unifying_algo_data = pd.read_parquet('../../../../Data/Unifying_Algo_Results/Main/' + fname)

In [ ]:
ax = layout.axes[('flash_course', 'laminar')]

# build aligned arrays
results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results

# build flash arrays from aligned time
flash_length = 675 # ms
aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
aligned_flash_arr[ix] = 1

# Model line (pink): built from the median fit parameters, shown from aligned t = -1 s on
model_line = np.atleast_2d(parametric_model(unifying_algo_data, ix_master))
model_line[:, np.asarray(ix_master) < MODEL_START_IX] = np.nan

# make the plot
uap.plot_aligned_course_from_array(ix_master, 
                                   aligned_time_since_flash_arr, 
                                   aligned_course_arr, 
                                   aligned_roi_arr,
                                   aligned_flash_arr=None,
                                   aligned_linear_fits=None, # if None, skips plot
                                   aligned_affine_fits=model_line, # if None, skips plot
                                   xlim_start=-300,
                                   xlim_end=300,
                                   ax=ax,
                                   clean_spines=True,
                                   course_marker_size=COURSE_MARKER_SIZE,
                                   course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                   )

In [ ]:
item = layout.svgitems['text_n_laminar']
N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
item.text = 'n=' + str(N_trajecs)

In [ ]:
clean_labels(ax, show_labels)

# With real unsteady data

In [ ]:
show_labels = False

In [ ]:
df = pd.read_hdf('../../../../Data/Experimental_Fly_Data/flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed.hdf')
fname = 'unsteady_top_on_data_flash_translation' + str(TRANSLATION) + '.parquet'
unifying_algo_data = pd.read_parquet('../../../../Data/Unifying_Algo_Results/Main/' + fname)

In [ ]:
ax = layout.axes[('flash_course', 'unsteady')]

# build aligned arrays
results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results

# build flash arrays from aligned time
flash_length = 675 # ms
aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
aligned_flash_arr[ix] = 1

# Model line (pink): built from the median fit parameters, shown from aligned t = -1 s on
model_line = np.atleast_2d(parametric_model(unifying_algo_data, ix_master))
model_line[:, np.asarray(ix_master) < MODEL_START_IX] = np.nan

# make the plot
uap.plot_aligned_course_from_array(ix_master, 
                                   aligned_time_since_flash_arr, 
                                   aligned_course_arr, 
                                   aligned_roi_arr,
                                   aligned_flash_arr=None,
                                   aligned_linear_fits=None, # if None, skips plot
                                   aligned_affine_fits=model_line, # if None, skips plot
                                   xlim_start=-300,
                                   xlim_end=300,
                                   ax=ax,
                                   clean_spines=True,
                                   course_marker_size=COURSE_MARKER_SIZE,
                                   course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                   )

In [ ]:
item = layout.svgitems['text_n_unsteady']
N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
item.text = 'n=' + str(N_trajecs)

In [ ]:
clean_labels(ax, show_labels)

# With real stillair data

In [ ]:
show_labels = True

In [ ]:
df = pd.read_hdf('../../../../Data/Experimental_Fly_Data/flies_stillair_c1xwt_preprocessed_optotrigger_trimmed.hdf')
fname = 'stillair_data_flash_translation' + str(TRANSLATION) + '.parquet'
unifying_algo_data = pd.read_parquet('../../../../Data/Unifying_Algo_Results/Main/' + fname)

In [ ]:
ax = layout.axes[('flash_course', 'stillair')]

# build aligned arrays
results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results

# build flash arrays from aligned time
flash_length = 675 # ms
aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
aligned_flash_arr[ix] = 1

# Model line (pink): built from the median fit parameters, shown from aligned t = -1 s on
model_line = np.atleast_2d(parametric_model(unifying_algo_data, ix_master))
model_line[:, np.asarray(ix_master) < MODEL_START_IX] = np.nan

# make the plot
uap.plot_aligned_course_from_array(ix_master, 
                                   aligned_time_since_flash_arr, 
                                   aligned_course_arr, 
                                   aligned_roi_arr,
                                   aligned_flash_arr=None,
                                   aligned_linear_fits=None, # if None, skips plot
                                   aligned_affine_fits=model_line, # if None, skips plot
                                   xlim_start=-300,
                                   xlim_end=300,
                                   ax=ax,
                                   clean_spines=True,
                                   course_marker_size=COURSE_MARKER_SIZE,
                                   course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                   )

In [ ]:
item = layout.svgitems['text_n_stillair']
N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
item.text = 'n=' + str(N_trajecs)

In [ ]:
clean_labels(ax, show_labels)

# With real low wind data

In [ ]:
show_labels = False

In [ ]:

#fname = 'varwind_N200_28.0_translation' + str(TRANSLATION) + '.parquet'


df = pd.read_parquet('../../../../Data/Experimental_Fly_Data/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet')

fname = 'varwind_N700_flash_28.0_translation' + str(TRANSLATION) + '.parquet'
unifying_algo_data = pd.read_parquet('../../../../Data/Unifying_Algo_Results/Supplemental/' + fname)

In [ ]:
np.random.seed(42)
obj_ids = np.random.choice(unifying_algo_data.obj_id_unique_event.unique(), size=200, replace=False)
unifying_algo_data = unifying_algo_data[unifying_algo_data.obj_id_unique_event.isin(obj_ids)]

In [ ]:
ax = layout.axes[('flash_course', 'low')]

# build aligned arrays
results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results

# build flash arrays from aligned time
flash_length = 675 # ms
aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
aligned_flash_arr[ix] = 1

# Model line (pink): built from the median fit parameters, shown from aligned t = -1 s on
model_line = np.atleast_2d(parametric_model(unifying_algo_data, ix_master))
model_line[:, np.asarray(ix_master) < MODEL_START_IX] = np.nan

# make the plot
uap.plot_aligned_course_from_array(ix_master, 
                                   aligned_time_since_flash_arr, 
                                   aligned_course_arr, 
                                   aligned_roi_arr,
                                   aligned_flash_arr=None,
                                   aligned_linear_fits=None, # if None, skips plot
                                   aligned_affine_fits=model_line, # if None, skips plot
                                   xlim_start=-300,
                                   xlim_end=300,
                                   ax=ax,
                                   clean_spines=True,
                                   course_marker_size=COURSE_MARKER_SIZE,
                                   course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                   )

In [ ]:
item = layout.svgitems['text_n_low']
N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
item.text = 'n=' + str(N_trajecs)

In [ ]:
clean_labels(ax, show_labels)

In [ ]:
layout.apply_svg_attrs()

layout.append_figure_to_layer(layout.figures['flash_course'], 'flash_course', cleartarget=True)
layout.write_svg(FIGURE_NAME)

# Sham data (row D)

Same plots as panel C, for the sham (no flash) trials of each condition. Written to its own figurefirst figure / layer, `sham_aligned_course`.

In [ ]:
def plot_aligned_course_panel(ax, unifying_algo_data, df, show_labels):
    """Aligned course vs time with the pink model line, exactly as the panel C cells above."""
    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results

    # Model line (pink): built from the median fit parameters, shown from aligned t = -1 s on
    model_line = np.atleast_2d(parametric_model(unifying_algo_data, ix_master))
    model_line[:, np.asarray(ix_master) < MODEL_START_IX] = np.nan

    uap.plot_aligned_course_from_array(ix_master,
                                       aligned_time_since_flash_arr,
                                       aligned_course_arr,
                                       aligned_roi_arr,
                                       aligned_flash_arr=None,
                                       aligned_linear_fits=None, # if None, skips plot
                                       aligned_affine_fits=model_line, # if None, skips plot
                                       xlim_start=-300,
                                       xlim_end=300,
                                       ax=ax,
                                       clean_spines=True,
                                       course_marker_size=COURSE_MARKER_SIZE,
                                       course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                       )
    clean_labels(ax, show_labels)


In [ ]:
DATA_DIR = '../../../../Data/'

# condition -> (trajectory file, unifying results file, show axis labels)
SHAM_DATA = {
    'stillair': ('Experimental_Fly_Data/flies_stillair_c1xwt_preprocessed_optotrigger_trimmed.hdf',
                 'Unifying_Algo_Results/Main/stillair_data_sham_translation' + str(TRANSLATION) + '.parquet', True),
    'unsteady': ('Experimental_Fly_Data/flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed_SHAM_AUGMENTED_merged.parquet',
                 'Unifying_Algo_Results/Main/unsteady_top_on_data_sham_AUGMENTED_translation' + str(TRANSLATION) + '.parquet', False),
    # NB: the varwind_N700_sham_* files are fits to *flash* trajectories (all their obj ids are
    # constant_flash_500ms events), so the real sham fits come from var_wind_kevin_analyzed (fan setting 28 = 20 cm/s),
    # as in the variable wind supplemental figure.
    'low':      ('Experimental_Fly_Data/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet',
                 'unifying_algo_results/var_wind_kevin_analyzed/var-wind-28-fan_0-2-cms_all-traj_align_SHAM.parquet', False),
    'laminar':  ('Experimental_Fly_Data/flies_laminar_c1xwt_preprocessed_optotrigger_trimmed.hdf',
                 'Unifying_Algo_Results/Main/laminar_data_sham_translation' + str(TRANSLATION) + '.parquet', False),
}


In [ ]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

for condition, (trajec_fname, unifying_fname, show_labels) in SHAM_DATA.items():
    if trajec_fname.endswith('.hdf'):
        df = pd.read_hdf(DATA_DIR + trajec_fname)
    else:
        df = pd.read_parquet(DATA_DIR + trajec_fname)
    unifying_algo_data = pd.read_parquet(DATA_DIR + unifying_fname)
    # guard: sham panels must only contain sham trajectories. Sham is marked by trigger_description 'sham'
    # (variable wind) or by a zero flash intensity (still air / laminar). The augmented unsteady sham file has
    # neither column; it is sham by construction.
    events = df[df.obj_id_unique_event.isin(unifying_algo_data.obj_id_unique_event.unique())].groupby('obj_id_unique_event').first()
    if 'trigger_description' in events.columns and events.trigger_description.str.contains('sham').any():
        is_sham = events.trigger_description.str.contains('sham')
    elif 'intensity' in events.columns:
        is_sham = events.intensity == 0
    else:
        is_sham = None
    if is_sham is not None:
        print(condition, 'sham trajectories: %d of %d' % (is_sham.sum(), len(is_sham)))
        assert is_sham.all(), condition + ': results file contains non-sham trajectories'

    ax = layout.axes[('sham_aligned_course', condition)]
    plot_aligned_course_panel(ax, unifying_algo_data, df, show_labels)

    N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
    layout.svgitems['text_n_sham_' + condition].text = 'n=' + str(N_trajecs)
    print(condition, N_trajecs)


In [ ]:
layout.apply_svg_attrs()

layout.append_figure_to_layer(layout.figures['sham_aligned_course'], 'sham_aligned_course', cleartarget=True)
layout.write_svg(FIGURE_NAME)


In [ ]:
#data = diagnose_axis_elements(ax)